# ANLP — Session 1: Prompting baselines + BART fine-tune

**What this run does**
1. Pretrained zero/few/CoT prompting baselines for **BART, FLAN-T5, LED** (9 conditions).
2. Fine-tune BART chunk-merger from `facebook/bart-large-cnn` (2-phase, encoder freeze → unfreeze).
3. Inference with the fine-tuned BART checkpoint on 9 test matches.
4. ROUGE-only evaluation across every prediction JSON in `outputs/predictions/`.

**Estimated time:** ~2.5 h on T4. Prompting ~90 min total, BART training ~45 min, inference ~5 min.

**Setup before running:**
- Sidebar → Accelerator → **GPU T4 x2**
- Sidebar → Internet → **On**
- Save Version → **Save & Run All (Commit)** so outputs get snapshotted.

**Note:** Session 2 (LED fine-tune) is independent and can run in parallel.

In [ ]:
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
WORK = '/kaggle/working'
REPO = f'{WORK}/ANLP_Project'
if not os.path.isdir(REPO):
    !git clone --quiet --branch setup/local-run https://github.com/christiandalfarra/ANLP_Project.git $REPO
%cd $REPO
!git pull --quiet

In [ ]:
!pip install -q sentencepiece bert_score rouge_score 'transformers>=4.40' 'accelerate>=0.30'

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Wipe stale predictions and BART checkpoint

In [ ]:
!rm -rf checkpoints/bart outputs/predictions outputs/results
!mkdir -p outputs/predictions outputs/results

## All 9 prompting baselines (~90 min)

BART/FLAN/LED × zero/few/CoT. Each saves its own JSON; you can ctrl-C and the rest still works.

In [ ]:
!python scripts/run_prompting.py --all

## Fine-tune BART chunk-merger (~45 min)

Pre-generates per-chunk summaries with pretrained BART (cached to `checkpoints/_chunk_summary_cache/`), then trains BART to map concatenated chunk-summaries → gold reference. Two-phase: encoder frozen → unfrozen.

In [ ]:
!python scripts/run_finetuning.py --model bart --output_dir checkpoints/bart

## Fine-tuned BART inference (~5 min)

In [ ]:
!python scripts/run_inference_finetuned.py --model bart --checkpoint checkpoints/bart

## Evaluate (ROUGE-only — bert_score is buggy on Kaggle's transformers version)

In [ ]:
!python scripts/run_evaluation.py --no-bertscore

In [ ]:
import os, json, glob
for p in sorted(glob.glob('outputs/predictions/*.json')):
    print('===', os.path.basename(p))
    d = json.load(open(p))
    mid, summary = next(iter(d.items()))
    print(f'[{mid}]\n{summary[:500]}\n')
for p in sorted(glob.glob('outputs/results/*')):
    print(p)
    if p.endswith('.csv'):
        print(open(p).read())